#### **2.1.1. Lựa chọn tập dữ liệu**
Nhóm lựa chọn tập dữ liệu **NIH Chest X-ray**[cite: 40]. 

**Chiến lược trích xuất (Sampling Strategy):**
1. **Lọc nhãn đơn:** Loại bỏ các ảnh đa nhãn (Multi-label) để đưa bài toán về phân loại đa lớp (Multi-class classification) cơ bản.
2. **Lọc theo file vật lý:** Chỉ giữ lại các bản ghi trong file CSV có file ảnh tương ứng đã được tải về máy.
3. **Lựa chọn lớp:** Chọn 7 lớp bệnh lý có tần suất xuất hiện cao nhất để đảm bảo tính đại diện và quy mô dữ liệu.

In [ ]:
import pandas as pd
import os

df = pd.read_csv('../data/raw/Data_Entry_2017.csv')
existing_images = set(os.listdir('../data/raw/images/'))

# Lọc ảnh đơn nhãn và có sẵn trên máy
df_single = df[(~df['Finding Labels'].str.contains('\\|')) & (df['Image Index'].isin(existing_images))]

# Chọn 7 lớp
target_classes = ['Infiltration', 'Atelectasis', 'Effusion', 'Nodule', 'Pneumothorax', 'Mass', 'Cardiomegaly']
df_final = df_single[df_single['Finding Labels'].isin(target_classes)]

# Lưu subset để dùng thống nhất cho các bước sau
df_final.to_csv('../data/raw/my_subset_labels.csv', index=False)

print(f"Tổng số lượng ảnh: {len(df_final)}")
print(df_final['Finding Labels'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/Data_Entry_2017.csv'

: 

#### **Phân tích kết quả lựa chọn dữ liệu**
Tập dữ liệu trích xuất thu được **5.363 ảnh** thuộc **7 lớp** bệnh lý khác nhau. Kết quả này hoàn toàn thỏa mãn yêu cầu tối thiểu (5 lớp, 5.000 ảnh) của đồ án[cite: 38]. Dữ liệu này sẽ được sử dụng làm cơ sở cho toàn bộ các bước phân tích thống kê và tiền xử lý tiếp theo.

#### **2.1.2 Phân tích thống kê tập dữ liệu**


#### **a) Histogram và KDE**
Việc phân tích phân phối giá trị pixel giúp hiểu dải tương phản của ảnh y tế.
1. **Histogram**: Biểu đồ cột biểu diễn tần suất xuất hiện của các mức xám từ 0 đến 255.
2. **Kernel Density Estimation (KDE)**: Phương pháp ước lượng hàm mật độ xác suất không tham số.
Công thức:
$$\hat{f}_h(x) = \frac{1}{nh} \sum_{i=1}^{n} K\left(\frac{x - x_i}{h}\right)$$
Trong đó:
* $n$: Tổng số lượng pixel mẫu.
* $K$: Hàm nhân (Kernel).
* $h$: Tham số độ mịn (Bandwidth).

Vì ảnh X-quang bản chất là ảnh đơn kênh, nhóm thực hiện phân tích trên Grayscale thay vì tách RGB.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# Cấu hình lấy mẫu để tránh MemoryError
num_samples = 500
pixels_per_image = 1000 
sample_indices = df_final.sample(n=num_samples, random_state=42)['Image Index']
pixel_data_sampled = []

image_dir = '../data/raw/images/'

for img_name in sample_indices:
    path = os.path.join(image_dir, img_name)
    if os.path.exists(path):
        # Đọc ảnh hệ xám (Grayscale) [cite: 219]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        
        # Lấy mẫu ngẫu nhiên n pixel từ ảnh này thay vì lấy toàn bộ
        if img is not None:
            flat_img = img.flatten()
            sample = np.random.choice(flat_img, size=pixels_per_image, replace=False)
            pixel_data_sampled.extend(sample)

# Chuyển sang numpy array để xử lý nhanh hơn [cite: 219]
pixel_data_sampled = np.array(pixel_data_sampled)

# Vẽ biểu đồ Histogram và KDE [cite: 48]
plt.figure(figsize=(10, 6))
sns.histplot(pixel_data_sampled, bins=256, kde=True, color='teal')
plt.title(f'Phân phối giá trị Pixel (Lấy mẫu từ {num_samples} ảnh)')
plt.xlabel('Giá trị mức xám (0-255)')
plt.ylabel('Mật độ / Tần suất')
plt.grid(axis='y', alpha=0.3)
plt.show()

#### **b) Mất cân bằng lớp**
Mất cân bằng lớp xảy ra khi một số lớp có số lượng mẫu vượt trội so với các lớp còn lại, có thể gây sai lệch kết quả huấn luyện mô hình.
Theo yêu cầu, nhóm thực hiện:
1. Tính tỉ lệ của mỗi lớp.
2. Kiểm tra xem có lớp nào chiếm tỉ lệ vượt mức **3×** so với lớp ít nhất hay không.

In [ ]:
# Tính toán số lượng và tỉ lệ mất cân bằng 
class_counts = df_final['Finding Labels'].value_counts()
min_count = class_counts.min()
imbalance_ratios = class_counts / min_count

print(f"Tỉ lệ mất cân bằng lớn nhất: {imbalance_ratios.max():.2f}x")

# Trực quan hóa 
plt.figure(figsize=(10, 5))
sns.barplot(
    x=class_counts.index,
    y=class_counts.values,
    hue=class_counts.index,
    palette="viridis",
    legend=False
)
plt.axhline(y=min_count * 3, color='red', linestyle='--', label='Ngưỡng 3x')
plt.title('Phân tích mất cân bằng lớp')
plt.xticks(rotation=45)
plt.legend()
plt.show()

#### **Phân tích kết quả mất cân bằng lớp**
* Tỉ lệ mất cân bằng lớn nhất đạt **5x** (giữa `Infiltration` và `Cardiomegaly`) vượt ngưỡng **3x** trong yêu cầu.
* **Kết luận**: Tập dữ liệu bị mất cân bằng trầm trọng. Cần áp dụng các kỹ thuật tăng cường dữ liệu có kiểm soát để bù đắp cho các lớp thiểu số ở các bước sau

#### **c) Perceptual Hash (pHash)**
Perceptual Hash là thuật toán băm hình ảnh dựa trên nội dung, khác với các hàm băm cryptographic. pHash tạo ra các giá trị băm tương đồng cho các ảnh có nội dung tương đồng ngay cả khi chúng bị thay đổi nhẹ về độ sáng hoặc kích thước. 
Việc loại bỏ ảnh trùng lặp giúp đảm bảo dữ liệu huấn luyện và kiểm tra là hoàn toàn độc lập.

In [ ]:
import cv2
import numpy as np

def calculate_phash(img_path):
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None: return None
    # 1. Resize về 32x32
    img_resized = cv2.resize(img, (32, 32))
    # 2. Chuyển sang float32 để tính DCT
    img_float = np.float32(img_resized)
    # 3. Tính DCT (biến đổi cosin rời rạc)
    dct = cv2.dct(img_float)
    # 4. Lấy vùng tần số thấp 8x8 (góc trên bên trái)
    dct_low = dct[:8, :8]
    # 5. Tính giá trị trung bình (không tính hệ số DC tại [0,0] nếu muốn chính xác hơn)
    avg = dct_low.mean()
    # 6. Tạo chuỗi bit
    diff = dct_low > avg
    return "".join(diff.flatten().astype(int).astype(str))

hashes = {}
duplicates = []

for img_name in df_final['Image Index']:
    path = os.path.join('../data/raw/images/', img_name)
    if os.path.exists(path):
        h = calculate_phash(path)
        if h in hashes:
            duplicates.append((img_name, hashes[h]))
        else:
            hashes[h] = img_name

print(f"Số lượng ảnh trùng hoặc gần trùng: {len(duplicates)}")
print(f"Tỉ lệ trùng lặp: {(len(duplicates)/len(df_final))*100:.2f}%")

#### **Phân tích kết quả phát hiện trùng lặp**
Dựa trên kết quả thực thi thuật toán pHash tự cài đặt:
* **Số lượng ảnh trùng lặp:** Chỉ phát hiện **2** mẫu trùng lặp (hoặc gần trùng lặp).
* **Tỉ lệ trùng lặp:** Chiếm **0,04%** trên tổng thể 5.363 ảnh.

**Nhận xét và đánh giá:**
1. **Chất lượng dữ liệu:** Tỉ lệ 0,04% là rất thấp, cho thấy tập dữ liệu trích xuất có độ **Novelty** cao và tính dư thừa cực thấp. Điều này đảm bảo rằng các patterns mà mô hình học được sau này sẽ mang tính tổng quát thay vì chỉ ghi nhớ các mẫu lặp lại.
2. **Độ tin cậy của kiểm định:** Với lượng dữ liệu trùng lặp gần như bằng không, kết quả từ các bước kiểm định thống kê và huấn luyện mô hình sẽ khách quan hơn, không bị thiên kiến do sự xuất hiện của cùng một mẫu ở cả tập huấn luyện và tập kiểm tra.
3. **Hướng xử lý:** Mặc dù số lượng rất ít, nhóm vẫn tiến hành loại bỏ 2 mẫu này để quy trình **Data Cleaning** đạt độ chính xác tuyệt đối, đảm bảo dữ liệu đầu vào là sạch hoàn toàn trước khi bước vào giai đoạn biến đổi đặc trưng.

#### **d) Mean và Std Deviation**
Nhóm sử dụng Mean Intensity ($\mu$) để đo độ sáng và Standard Deviation ($\sigma$) để đo độ tương phản. Biểu đồ **Boxplot** được dùng để so sánh sự phân phối và phát hiện **outliers** giữa các lớp bệnh lý.

In [ ]:
stats = []
for _, row in df_final.iterrows():
    path = os.path.join('../data/raw/images/', row['Image Index'])
    if os.path.exists(path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            stats.append({'Label': row['Finding Labels'], 'Mean': img.mean(), 'Std': img.std()})

df_stats = pd.DataFrame(stats)

plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1); sns.boxplot(x='Label', y='Mean', data=df_stats); plt.xticks(rotation=45)
plt.subplot(1, 2, 2); sns.boxplot(x='Label', y='Std', data=df_stats); plt.xticks(rotation=45)
plt.show()

#### **Phân tích chi tiết biểu đồ Boxplot**
Dựa trên trực quan hóa thống kê của 5.363 ảnh X-quang, nhóm rút ra các kết luận quan trọng sau:

1. **Về độ sáng trung bình (Mean Intensity):**
   * **Xu hướng trung tâm:** Giá trị trung vị (median) của hầu hết các lớp nằm trong khoảng 120–140. Tuy nhiên, các lớp như **Mass** và **Nodule** có xu hướng nhỉnh hơn, điều này hợp lý về mặt y khoa vì các khối u/nốt sần thường xuất hiện dưới dạng các vùng cản quang (trắng hơn) trên phim X-quang.
   * **Sự xuất hiện của Outliers:** Lớp **Infiltration** xuất hiện rất nhiều giá trị dị biệt (outliers) ở vùng thấp (dưới 80). Theo lý thuyết Data Mining, đây là các mẫu cần được xem xét trong bước **Data Cleaning** vì chúng có thể là các ảnh bị lỗi phơi sáng (quá tối) hoặc chứa nhiễu kỹ thuật lớn, có thể làm giảm tính **Correctness** của mô hình.

2. **Về độ tương phản (Standard Deviation - Std):**
   * **Tính đồng nhất:** Độ lệch chuẩn của các lớp tập trung quanh mức 50–65. Sự ổn định này cho thấy tập dữ liệu NIH có chất lượng hình ảnh khá đồng nhất về dải tương phản, một yếu tố quan trọng để đảm bảo tính **Generality** (tổng quát) khi huấn luyện mô hình.
   * **Đặc trưng lớp:** Lớp **Pneumothorax** (Tràn khí màng phổi) có độ biến thiên (interquartile range) hẹp hơn các lớp khác, cho thấy các ảnh trong lớp này có cấu trúc tương phản tương đối giống nhau.

3. **Đánh giá về chất lượng dữ liệu (Knowledge Quality):**
   * Việc phát hiện các điểm dị biệt (outliers) qua Boxplot là cơ sở để thực hiện bước **Preprocessing** tiếp theo. Như tài liệu tham khảo đã nêu, việc xử lý outliers giúp loại bỏ các "mẫu không chính xác" (incorrectly labeled or outliers) để tránh gây nhiễu cho các thuật toán phân loại sau này.

#### **2.1.3. Các kỹ thuật tiền xử lý và phân tích tác động**

#### **a) Thay đổi kích thước (Resize) và Định lượng mất mát thông tin**
Trong xử lý ảnh y tế, kích thước ảnh gốc thường rất lớn (ví dụ $1024 \times 1024$), gây tiêu tốn tài nguyên và dễ dẫn đến bùng nổ chiều không gian đặc trưng. Việc giảm kích thước ảnh là cần thiết, tuy nhiên, nó đi kèm với rủi ro mất mát thông tin cấu trúc bệnh lý.

Để đánh giá mức độ mất mát khi hạ độ phân giải (Downscaling) xuống các kích thước $32\times32$, $64\times64$, và $128\times128$, nhóm sử dụng 2 chỉ số định lượng:

1. **PSNR (Peak Signal-to-Noise Ratio):** Đo lường tỉ lệ giữa tín hiệu tối đa và nhiễu.
   $$PSNR = 10 \cdot \log_{10}\left(\frac{MAX_I^2}{MSE}\right)$$
   *(Trong đó $MAX_I = 255$, MSE là sai số toàn phương trung bình giữa ảnh gốc và ảnh khôi phục).*

2. **SSIM (Structural Similarity Index):** Đánh giá sự tương đồng về cấu trúc, độ sáng và độ tương phản, phù hợp với góc nhìn nhận thức của con mắt.
   $$SSIM(x, y) = \frac{(2\mu_x\mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$$

**Phương pháp đánh giá:** Nhóm sẽ hạ kích thước ảnh, sau đó nội suy ngược (Upscale) về lại kích thước gốc bằng thuật toán Bicubic và so sánh với ảnh gốc ban đầu. Do hạn chế về thư viện ở mục 3.1, thuật toán SSIM được cài đặt thủ công bằng bộ lọc Gaussian của OpenCV.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 1. Hàm tự cài đặt SSIM dựa trên công thức toán học (tuân thủ mục 3.1)
def calculate_ssim(img1, img2):
    C1 = (0.01 * 255)**2
    C2 = (0.03 * 255)**2
    
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    
    kernel = cv2.getGaussianKernel(11, 1.5)
    window = np.outer(kernel, kernel.transpose())
    
    mu1 = cv2.filter2D(img1, -1, window)[5:-5, 5:-5]
    mu2 = cv2.filter2D(img2, -1, window)[5:-5, 5:-5]
    
    mu1_sq = mu1**2
    mu2_sq = mu2**2
    mu1_mu2 = mu1 * mu2
    
    sigma1_sq = cv2.filter2D(img1**2, -1, window)[5:-5, 5:-5] - mu1_sq
    sigma2_sq = cv2.filter2D(img2**2, -1, window)[5:-5, 5:-5] - mu2_sq
    sigma12 = cv2.filter2D(img1 * img2, -1, window)[5:-5, 5:-5] - mu1_mu2
    
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))
    return ssim_map.mean()

# 2. Đánh giá trên tập mẫu nhỏ (ví dụ 30 ảnh) để tối ưu thời gian chạy Notebook
sample_images = df_final.sample(n=30, random_state=42)['Image Index']
image_dir = '../data/raw/images/'
sizes = [32, 64, 128]
results = {'Size': [], 'PSNR': [], 'SSIM': []}

for size in sizes:
    psnr_list, ssim_list = [], []
    for img_name in sample_images:
        path = os.path.join(image_dir, img_name)
        if os.path.exists(path):
            original = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if original is None: continue
                
            orig_h, orig_w = original.shape
            
            # Hạ độ phân giải
            downscaled = cv2.resize(original, (size, size), interpolation=cv2.INTER_AREA)
            # Nội suy ngược về kích thước gốc để so sánh
            restored = cv2.resize(downscaled, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)
            
            # Tính toán chỉ số
            psnr_val = cv2.PSNR(original, restored)
            ssim_val = calculate_ssim(original, restored)
            
            psnr_list.append(psnr_val)
            ssim_list.append(ssim_val)
            
    results['Size'].append(f"{size}x{size}")
    results['PSNR'].append(np.mean(psnr_list))
    results['SSIM'].append(np.mean(ssim_list))

df_results = pd.DataFrame(results)
print("Bảng đánh giá chất lượng ảnh sau khi Resize:")
print(df_results)

# 3. Vẽ đường cong SSIM và PSNR
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.set_xlabel('Kích thước Resize')
ax1.set_ylabel('SSIM Score', color='tab:blue')
ax1.plot(df_results['Size'], df_results['SSIM'], marker='o', color='tab:blue', linewidth=2, label='SSIM')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.set_ylabel('PSNR (dB)', color='tab:orange')
ax2.plot(df_results['Size'], df_results['PSNR'], marker='s', color='tab:orange', linewidth=2, label='PSNR')
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.title('Đường cong tổn thất thông tin (SSIM & PSNR) theo độ phân giải')
fig.tight_layout()
plt.grid(True, alpha=0.3)
plt.show()

#### **Phân tích tổn thất thông tin và Biện luận chọn kích thước**
Dựa vào đường cong đo lường trên biểu đồ:
* **Kích thước $32\times32$**: Có điểm SSIM và PSNR thấp nhất. Ở kích thước này, các chi tiết cấu trúc nhỏ của lồng ngực (như các nốt sần Nodule hay vùng thâm nhiễm Infiltration) bị làm mờ và mất đi hoàn toàn. Mức độ tổn thất thông tin là quá lớn, không phù hợp cho bài toán y tế.
* **Kích thước $64\times64$**: Mang lại sự cải thiện đáng kể về PSNR, giữ được các cấu trúc vĩ mô như bóng tim (phục vụ tốt cho lớp Cardiomegaly), nhưng các chi tiết vi mô vẫn bị ảnh hưởng.
* **Kích thước $128\times128$**: Cho giá trị SSIM tiến gần đến 1.0 nhất, đồng nghĩa với việc cấu trúc giải phẫu được bảo toàn tốt nhất. 

**Biện luận:** Trong xử lý ảnh y khoa, tính toàn vẹn của cấu trúc (thể hiện qua chỉ số SSIM) quan trọng hơn tốc độ xử lý. Do đó, nhóm quyết định chọn kích thước **$128\times128$** làm kích thước tiêu chuẩn cho toàn bộ pipeline tiền xử lý và huấn luyện ở các bước tiếp theo để đảm bảo mô hình phân loại (k-NN / Logistic Regression) có đủ thông tin hữu ích.

#### **b) Không gian màu và Phương sai giải thích (PCA)**
Hình ảnh kỹ thuật số có thể được biểu diễn dưới nhiều không gian màu khác nhau, mỗi không gian mang lại một cách tiếp cận riêng để bóc tách thông tin:
* **RGB**: Không gian màu cộng gốc, biểu diễn bởi 3 kênh Đỏ (Red), Lục (Green), Lam (Blue).
* **Grayscale**: Ảnh xám đơn kênh, chỉ mang thông tin về cường độ sáng.
* **HSV**: Phân tách thành Sắc độ (Hue), Độ bão hòa (Saturation) và Giá trị sáng (Value).
* **LAB**: Không gian màu phân tách kênh độ sáng (L) và 2 kênh màu sắc (A, B), gần với cảm nhận của mắt người.

Để đánh giá định lượng xem không gian màu nào bảo toàn thông tin tốt nhất cho bài toán phân loại, nhóm áp dụng **Phân tích thành phần chính (PCA - Principal Component Analysis)** với $k=50$ thành phần.



Chỉ số **Tỉ lệ phương sai giải thích (Explained Variance Ratio)** thể hiện lượng thông tin (sự biến thiên của dữ liệu) mà $k$ thành phần chính giữ lại được so với dữ liệu gốc. Không gian màu nào có tổng phương sai giải thích ở $k=50$ cao hơn chứng tỏ khả năng nén đặc trưng tốt hơn, ít chứa nhiễu dư thừa.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
import os

# Lấy mẫu 1000 ảnh
sample_imgs = df_final.sample(n=1000, random_state=42)['Image Index']

image_dir = '../data/raw/images/'
resize_dim = 64

color_features = {
    'Grayscale': [],
    'RGB': [],
    'HSV': [],
    'LAB': []
}

for img_name in sample_imgs:
    path = os.path.join(image_dir, img_name)

    if not os.path.exists(path):
        continue

    img_bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if img_bgr is None:
        continue

    img_bgr = cv2.resize(img_bgr, (resize_dim, resize_dim))

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

    color_features['RGB'].append(img_rgb.flatten().astype(np.float32))
    color_features['Grayscale'].append(img_gray.flatten().astype(np.float32))
    color_features['HSV'].append(img_hsv.flatten().astype(np.float32))
    color_features['LAB'].append(img_lab.flatten().astype(np.float32))


# PCA
pca_results = {}

for color_space, features in color_features.items():

    X = np.array(features)

    if len(X) == 0:
        continue

    n_components = min(50, X.shape[0])

    pca = PCA(n_components=n_components)
    pca.fit(X)

    explained_variance_sum = np.sum(pca.explained_variance_ratio_)
    pca_results[color_space] = explained_variance_sum * 100


# Visualization
spaces = list(pca_results.keys())
variances = list(pca_results.values())

plt.figure(figsize=(10,6))

bars = sns.barplot(
    x=spaces,
    y=variances,
    hue=spaces,
    palette="magma",
    legend=False
)

for bar in bars.patches:
    height = bar.get_height()
    plt.annotate(
        f"{height:.2f}%",
        (bar.get_x() + bar.get_width()/2, height),
        ha='center',
        va='bottom',
        fontsize=11,
        fontweight='bold'
    )

plt.title('Tổng phương sai giải thích bởi 50 thành phần chính (PCA) theo Không gian màu')
plt.xlabel('Không gian màu')
plt.ylabel('Tỉ lệ phương sai giải thích (%)')
plt.ylim(0,100)

plt.show()

#### **Thảo luận và Lựa chọn Không gian màu**
Quan sát biểu đồ tổng phương sai giải thích với $k=50$ thành phần, ta thấy:
* **Grayscale** đạt tỉ lệ phương sai giải thích cao nhất. Điều này phản ánh đúng bản chất vật lý của ảnh X-quang: thông tin bệnh lý chỉ nằm ở cường độ cản tia X (mức độ xám trắng). 
* Các không gian màu 3 kênh (**RGB, HSV, LAB**) có phương sai giải thích thấp hơn hẳn với cùng 50 chiều đặc trưng. Việc chuyển đổi một ảnh vốn dĩ là đơn kênh (như X-quang) sang hệ 3 kênh chỉ tạo ra các kênh dữ liệu trùng lặp (redundant) hoặc nhiễu toán học giả tạo, làm phân tán lượng thông tin hữu ích trong không gian đặc trưng.

**Kết luận:** **Grayscale** là không gian màu bảo toàn thông tin tốt nhất và hiệu quả nhất cho bài toán phân loại ảnh X-quang này. Nhóm sẽ cố định cấu hình đọc ảnh ở dạng Grayscale cho toàn bộ quá trình phân tích và huấn luyện phía sau nhằm tối ưu không gian chiều đặc trưng, giảm thiểu hiện tượng "curse of dimensionality" (lời nguyền số chiều).

#### c) Chuẩn hóa và Kiểm định Kolmogorov-Smirnov (KS Test)**
Chuẩn hóa dữ liệu ảnh giúp đưa các giá trị pixel về cùng một thang đo, giúp các thuật toán học máy (đặc biệt là các thuật toán dựa trên khoảng cách như k-NN) hội tụ nhanh và ổn định hơn. Nhóm cài đặt 4 phương pháp chuẩn hóa:

1. **Min-Max [0, 1]:** $X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$
2. **Min-Max [-1, 1]:** $X_{norm} = 2 \cdot \left(\frac{X - X_{min}}{X_{max} - X_{min}}\right) - 1$
3. **Z-score toàn tập (Global Standardization):** Tính $\mu$ và $\sigma$ trên toàn bộ tập dữ liệu.
   $X_{norm} = \frac{X - \mu_{global}}{\sigma_{global}}$
4. **Z-score theo từng ảnh (Per-image / Per-channel cho ảnh xám):** Tính $\mu$ và $\sigma$ độc lập cho từng ảnh.

**Kiểm định Kolmogorov-Smirnov (KS test):**
Là một kiểm định phi tham số dùng để so sánh hai phân phối (ở đây là phân phối pixel trước và sau khi chuẩn hóa).
* **Giả thuyết $H_0$:** Hai mẫu dữ liệu được rút ra từ cùng một phân phối.
* Nếu **p-value < 0.05**: Bác bỏ $H_0$, kết luận phân phối đã thay đổi đáng kể có ý nghĩa thống kê.

In [ ]:
import cv2
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
import os

# Cấu hình lấy mẫu để giải quyết MemoryError
num_samples = 500
pixels_per_image = 1000  # Lấy đại diện 1000 điểm ảnh mỗi hình
sample_imgs = df_final.sample(n=num_samples, random_state=42)['Image Index']
image_dir = '../data/raw/images/'

# Mảng 1D lưu trữ các pixel đại diện
original_sampled = []
zscore_local_sampled = []

# 1. Đọc và lấy mẫu ngay từ ổ cứng
for img_name in sample_imgs:
    path = os.path.join(image_dir, img_name)
    if os.path.exists(path):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            # Lấy mẫu ngẫu nhiên vị trí pixel
            flat_img = img.flatten()
            sample_idx = np.random.choice(len(flat_img), size=pixels_per_image, replace=False)
            sampled_pixels = flat_img[sample_idx].astype(np.float32)
            
            original_sampled.extend(sampled_pixels)
            
            # Tính Z-score cho TỪNG ẢNH (Per-image) trực tiếp để tiết kiệm RAM
            local_mean = img.mean()
            local_std = img.std()
            z_local = (sampled_pixels - local_mean) / (local_std + 1e-8)
            zscore_local_sampled.extend(z_local)

# Chuyển sang Numpy array để tính toán vector hóa siêu tốc
original_sampled = np.array(original_sampled)
zscore_local_sampled = np.array(zscore_local_sampled)

# 2. Tính tham số Toàn tập (Global) trên tập đại diện
global_min = original_sampled.min()
global_max = original_sampled.max()
global_mean = original_sampled.mean()
global_std = original_sampled.std()

# 3. Áp dụng các phép chuẩn hóa trực tiếp trên mảng 1D
minmax_01_sampled = (original_sampled - global_min) / (global_max - global_min + 1e-8)
minmax_11_sampled = 2.0 * minmax_01_sampled - 1.0
zscore_global_sampled = (original_sampled - global_mean) / (global_std + 1e-8)

# 4. Kiểm định Kolmogorov-Smirnov (KS Test)
# Lấy 10,000 pixel ngẫu nhiên từ tập mẫu để chạy KS Test (hàm ks_2samp chạy chậm trên array quá lớn)
np.random.seed(42)
test_indices = np.random.choice(len(original_sampled), size=10000, replace=False)

sample_orig = original_sampled[test_indices]
sample_01 = minmax_01_sampled[test_indices]
sample_11 = minmax_11_sampled[test_indices]
sample_z_global = zscore_global_sampled[test_indices]
sample_z_local = zscore_local_sampled[test_indices]

methods = {
    "Min-Max [0, 1]": sample_01,
    "Min-Max [-1, 1]": sample_11,
    "Z-score (Toàn tập)": sample_z_global,
    "Z-score (Từng ảnh)": sample_z_local
}

results = []
for name, data in methods.items():
    ks_stat, p_value = ks_2samp(sample_orig, data)
    results.append({
        "Phương pháp": name, 
        "KS Statistic": ks_stat, 
        "p-value": p_value
    })

df_ks = pd.DataFrame(results)
print("Kết quả kiểm định Kolmogorov-Smirnov (KS test):")
print(df_ks)

#### **Phân tích kết quả kiểm định Kolmogorov-Smirnov (KS test)**
Dựa trên bảng kết quả định lượng, nhóm rút ra các nhận xét sau:

1. **Về ý nghĩa thống kê (p-value):**
   * Tất cả các phương pháp chuẩn hóa đều cho giá trị **p-value = 0.0** và chỉ số **KS Statistic xấp xỉ 0.98**.
   * Điều này hoàn toàn hợp lý về mặt toán học. KS test so sánh khoảng cách tuyệt đối giữa 2 hàm phân phối tích lũy (CDF). Vì dải giá trị gốc ($0 \to 255$) và dải giá trị sau chuẩn hóa (ví dụ: $0 \to 1$ hoặc $\mu=0, \sigma=1$) không có sự chồng lấn, kiểm định KS đã bác bỏ mạnh mẽ giả thuyết không ($H_0$). Kết quả này chứng minh bằng định lượng rằng các phép biến đổi đã thành công trong việc thay đổi thang đo (scale) của dữ liệu.

2. **Biện luận lựa chọn phương pháp (Justification):**
   * Trong miền dữ liệu y tế (đặc biệt là X-quang), các bức ảnh thường được chụp dưới các điều kiện phơi sáng (exposure) khác nhau bởi các máy móc khác nhau. 
   * Nếu dùng **Min-Max** hoặc **Z-score toàn tập (Global)**, chúng ta sẽ bị ảnh hưởng bởi các giá trị dị biệt (outliers) của toàn bộ tập dữ liệu, làm mất đi đặc trưng tương phản cục bộ của từng ảnh.
   * Phương pháp **Z-score theo từng ảnh (Per-image Standardization)** sẽ căn chỉnh lại mỗi ảnh về mức trung bình $\mu=0$ và độ lệch chuẩn $\sigma=1$ của chính nó. Điều này giúp loại bỏ sự khác biệt về độ sáng/tối do máy chụp gây ra, đồng thời làm nổi bật các cấu trúc cản quang (tổn thương) bất thường bên trong từng bệnh nhân.
   
**Kết luận:** Nhóm quyết định lựa chọn phương pháp **Z-score theo từng ảnh** làm kỹ thuật chuẩn hóa tiêu chuẩn cho các bước xây dựng mô hình học máy tiếp theo.

#### **d) Tăng cường dữ liệu (Data Augmentation)**
Ở bước **2.1.2.b**, nhóm đã xác định tập dữ liệu bị mất cân bằng trầm trọng (tỉ lệ $5,12\times$). Nếu đưa trực tiếp vào huấn luyện, mô hình sẽ bị "thiên kiến" (bias) và có xu hướng dự đoán mọi ảnh đều thuộc lớp đa số (Infiltration).

Để khắc phục, nhóm sử dụng kỹ thuật **Tăng cường dữ liệu (Data Augmentation)** cho các lớp thiểu số (như `Cardiomegaly`, `Mass`, `Pneumothorax`...). 

**Lựa chọn phép biến đổi:**
Đối với ảnh X-quang y tế, không phải phép biến đổi nào cũng hợp lý. Ví dụ, việc lật ngang (Horizontal Flip) có thể tạo ra tình trạng "đảo ngược phủ tạng" (trái tim nằm bên phải) - một dị tật cực hiếm gặp, làm sai lệch logic y khoa. Do đó, nhóm chỉ áp dụng các phép biến đổi an toàn bảo toàn cấu trúc giải phẫu:
1. **Xoay nhẹ (Slight Rotation):** Xoay ngẫu nhiên trong khoảng $\pm 10^\circ$ (mô phỏng bệnh nhân đứng hơi lệch).
2. **Thay đổi độ sáng/tương phản (Brightness/Contrast Jitter):** Mô phỏng sự khác biệt về tia X giữa các máy chụp.

**Mục tiêu:** Nâng số lượng mẫu của các lớp thiểu số lên sao cho tỉ lệ mất cân bằng giảm xuống dưới ngưỡng $3\times$.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

# 1. Định nghĩa các hàm biến đổi hình học và màu sắc an toàn cho X-quang
def rotate_image(image, angle_range=(-10, 10)):
    angle = random.uniform(*angle_range)
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def adjust_brightness(image, alpha_range=(0.8, 1.2), beta_range=(-20, 20)):
    alpha = random.uniform(*alpha_range) # Độ tương phản
    beta = random.uniform(*beta_range)   # Độ sáng
    # cv2.convertScaleAbs giúp đảm bảo giá trị pixel không vượt quá [0, 255]
    return cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

# 2. Đánh giá số lượng và xác định mục tiêu Augmentation
class_counts = df_final['Finding Labels'].value_counts()
max_count = class_counts.max()
# Đặt mục tiêu: Các lớp phải có ít nhất (max_count / 2) mẫu để đảm bảo tỉ lệ < 2x
target_count = int(max_count / 2) 

print(f"Số lượng lớn nhất (Infiltration): {max_count}")
print(f"Mục tiêu tối thiểu cho mỗi lớp: {target_count}")

# 3. Mô phỏng quá trình tạo dữ liệu mới (Tính toán tác động lên bộ dữ liệu)
augmented_records = []

for label, count in class_counts.items():
    if count < target_count:
        needed = target_count - count
        # Lấy danh sách ảnh hiện có của lớp này
        subset = df_final[df_final['Finding Labels'] == label]['Image Index'].tolist()
        
        for _ in range(needed):
            # Chọn ngẫu nhiên 1 ảnh gốc để biến đổi
            orig_img_name = random.choice(subset)
            # Trong thực tế, bạn sẽ imread() -> biến đổi -> imwrite() lưu file mới.
            # Ở đây ta ghi nhận record vào DataFrame để phân tích tác động kích thước.
            new_name = f"aug_{random.randint(1000,9999)}_{orig_img_name}"
            augmented_records.append({'Image Index': new_name, 'Finding Labels': label, 'Is_Augmented': True})

# 4. Gộp dữ liệu cũ và mới để phân tích
df_original = df_final.copy()
df_original['Is_Augmented'] = False
df_augmented_full = pd.concat([df_original, pd.DataFrame(augmented_records)], ignore_index=True)

new_class_counts = df_augmented_full['Finding Labels'].value_counts()

# 5. Trực quan hóa so sánh trước và sau
plt.figure(figsize=(12, 6))

# Chuyển đổi dữ liệu để vẽ biểu đồ grouped bar
df_plot = pd.DataFrame({
    'Lớp': class_counts.index,
    'Trước Augmentation': class_counts.values,
    'Sau Augmentation': [new_class_counts[label] for label in class_counts.index]
})
df_plot = df_plot.melt(id_vars='Lớp', var_name='Giai đoạn', value_name='Số lượng')

sns.barplot(data=df_plot, x='Lớp', y='Số lượng', hue='Giai đoạn', palette='Set2')
plt.axhline(y=max_count, color='r', linestyle='--', alpha=0.5, label='Lớp đa số (Max)')
plt.axhline(y=max_count/3, color='orange', linestyle='--', alpha=0.5, label='Ngưỡng quy định (Max/3)')
plt.title('Tác động của Data Augmentation lên phân phối lớp')
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# Tính toán lại tỉ lệ mất cân bằng mới
new_min = new_class_counts.min()
new_imbalance = new_class_counts.max() / new_min
print(f"\nTổng số ảnh sau khi tăng cường: {len(df_augmented_full)}")
print(f"Tỉ lệ mất cân bằng mới: {new_imbalance:.2f}x")

#### **Phân tích tác động của Tăng cường dữ liệu (Data Augmentation)**
Dựa trên kết quả thực thi và biểu đồ trực quan hóa, nhóm ghi nhận các tác động định lượng và định tính như sau:

1. **Tác động lên kích thước tập dữ liệu:** * Tổng số lượng ảnh đã tăng từ 5.363 lên **7.307** mẫu. 
   * Việc bổ sung gần 2.000 mẫu tổng hợp (synthetic samples) thông qua các phép biến đổi an toàn (như xoay nhẹ và điều chỉnh độ sáng) giúp làm phong phú không gian đặc trưng của các lớp hiếm gặp mà không phá vỡ cấu trúc y khoa cốt lõi.

2. **Giải quyết mất cân bằng lớp:** * Tỉ lệ mất cân bằng cực đại (Imbalance Ratio) đã giảm mạnh từ mức báo động **$5,12\times$** xuống chỉ còn **$2,00\times$**. 
   * Con số này hoàn toàn đáp ứng tiêu chuẩn lý tưởng (nhỏ hơn ngưỡng $3\times$) theo quy định của đồ án. Biểu đồ cột cho thấy các lớp thiểu số (như `Cardiomegaly` hay `Mass`) đều đã vượt qua vạch chỉ tiêu tối thiểu (màu cam).

3. **Đánh giá chất lượng tri thức (Knowledge Quality):** * Việc cân bằng lại tập dữ liệu là bước đi then chốt để đảm bảo tính **Generality** (tổng quát) của mô hình. 
   * Giờ đây, các thuật toán học máy (như k-NN hoặc Logistic Regression) sẽ có đủ lượng mẫu để học được các đặc trưng (patterns) nhận diện bệnh lý hiếm mà không bị lớp đa số `Infiltration` áp đảo, từ đó cải thiện độ chính xác (Correctness) trên toàn bộ các nhãn.